# SCOPE — fine-tuning the Vision Transformer

Trains the evidence classifier used at **S10**: given a photograph, return
P(smoking violation).

**Run this in Google Colab with a GPU runtime.** Runtime → Change runtime type →
T4 GPU. On CPU a single epoch takes hours; on a T4 it takes minutes.

### Why KerasHub and not HuggingFace `transformers`

`transformers` **removed TensorFlow support** — `TFViTForImageClassification`
no longer exists. KerasHub is the Keras-native replacement, runs on the
TensorFlow backend, and keeps the project honest to Table 3.5 of the report
(*TensorFlow / Keras*). Nothing here needs PyTorch.

### Before you start

Run `python ml/prepare_dataset.py` locally first. This notebook consumes the
`manifest.csv` it produces — the split is decided there, deliberately, so that
it is reproducible and free of train/test leakage rather than re-rolled here.


## 1. Environment


In [ ]:
!pip install -q keras-hub keras --upgrade

import os
os.environ['KERAS_BACKEND'] = 'tensorflow'   # Table 3.5 commits to TensorFlow

import json, pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub

print('keras     ', keras.__version__)
print('tensorflow', tf.__version__)
print('GPU       ', tf.config.list_physical_devices('GPU') or 'NONE — switch runtime to T4 GPU')


## 2. Data

Upload a zip containing `manifest.csv` and the `data/raw/` tree, or mount Drive.
The manifest's `path` column is relative to the `ml/` directory.


In [ ]:
from google.colab import files

# Option A — upload a zip of the ml/data directory
uploaded = files.upload()          # choose e.g. scope-data.zip
!unzip -q -o $(ls *.zip | head -1) -d /content/ml

# Option B — Google Drive instead (comment out Option A above)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/scope/ml' /content/ml

ML_DIR = pathlib.Path('/content/ml')
manifest = pd.read_csv(ML_DIR / 'data' / 'manifest.csv')
print(manifest.groupby(['split', 'label']).size())


### Verify the split before training

Cheap, and it protects the headline result: if a perceptual hash appeared in
two splits, the reported test accuracy would be inflated by memorisation.


In [ ]:
spanning = manifest.groupby('phash')['split'].nunique()
assert (spanning <= 1).all(), 'LEAKAGE: a hash group spans multiple splits'
assert manifest['path'].is_unique, 'duplicate paths in manifest'
print('No leakage between splits.')
print('Total images:', len(manifest))


## 3. Input pipeline

Augmentation is applied to the **training split only** — augmenting validation
or test would measure the model against images it will never see in the field.

There is no vertical flip: a cigarette has an orientation, and an upside-down
photograph is not a case this app will ever encounter.


In [ ]:
IMAGE_SIZE = 224          # ViT base patch16 expects 224x224
BATCH_SIZE = 32
CLASS_NAMES = ['notsmoking', 'smoking']   # index 1 == positive class
AUTOTUNE = tf.data.AUTOTUNE


def decode(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    return tf.cast(img, tf.float32), label


def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0), label


def build(split, training):
    rows = manifest[manifest['split'] == split]
    paths = [str(ML_DIR / p) for p in rows['path']]
    labels = [CLASS_NAMES.index(l) for l in rows['label']]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=20260813, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = build('train', training=True)
val_ds   = build('val',   training=False)
test_ds  = build('test',  training=False)

images, labels = next(iter(train_ds))
print('batch', images.shape, 'range', float(tf.reduce_min(images)), '-', float(tf.reduce_max(images)))


## 4. Model

A ViT pre-trained on ImageNet-21k, with a fresh 2-class head. The dataset here
is far too small to train a transformer from scratch — transfer learning is what
makes this feasible at all, and is the approach Mazurets et al. (2022) used for
the same task.


In [ ]:
PRESET = 'vit_base_patch16_224_imagenet21k'

# activation='softmax' is REQUIRED and is not the default.
#
# KerasHub ships activation=None, which makes the model output raw LOGITS —
# unbounded numbers that can be negative. Pairing that with
# SparseCategoricalCrossentropy(from_logits=False) silently trains against a
# broken objective: the loss treats negative logits as probabilities, learns
# almost nothing, and predict() then returns negative 'confidence' scores that
# no threshold can work with.
#
# Two consistent options exist. This notebook takes the first:
#   activation='softmax'  +  from_logits=False   <- outputs are probabilities
#   activation=None       +  from_logits=True    <- outputs are logits
classifier = keras_hub.models.ViTImageClassifier.from_preset(
    PRESET,
    num_classes=2,
    activation='softmax',
)

# Stage 1: freeze the backbone. The new head starts random, and letting large
# random gradients flow into pre-trained weights destroys the features that
# make transfer learning work.
classifier.backbone.trainable = False

classifier.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

# Fail loudly here rather than discovering it after a full training run.
_probe = classifier.predict(val_ds.take(1), verbose=0)
assert _probe.min() >= 0.0 and _probe.max() <= 1.0, (
    f'Model is emitting logits, not probabilities (range {_probe.min():.3f} to '
    f'{_probe.max():.3f}). Check the activation argument above.')
import numpy as _np
assert _np.allclose(_probe.sum(axis=1), 1.0, atol=1e-3), 'Rows should sum to 1.'
print(f'Output check passed: probabilities in [{_probe.min():.3f}, {_probe.max():.3f}]')
classifier.summary()


## 5. Stage 1 — train the head


In [ ]:
EPOCHS_HEAD = 5

history_head = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True),
    ],
)


## 6. Stage 2 — fine-tune

Unfreeze the backbone at a much lower learning rate. 1e-3 would wreck the
pre-trained weights; 1e-5 adapts them.


In [ ]:
classifier.backbone.trainable = True

classifier.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

EPOCHS_FINE = 10
history_fine = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
    ],
)


## 7. Training curves

Chapter 5 needs these. A validation loss that rises while training loss keeps
falling is overfitting — collect more data or stop earlier.


In [ ]:
import matplotlib.pyplot as plt

def series(key):
    return list(history_head.history[key]) + list(history_fine.history[key])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (a, b, title) in zip(axes, [('accuracy','val_accuracy','Accuracy'),
                                    ('loss','val_loss','Loss')]):
    ax.plot(series(a), label='train')
    ax.plot(series(b), label='validation')
    ax.axvline(len(history_head.history[a]) - 0.5, ls='--', c='grey', lw=1)
    ax.set_title(title + '  (dashed = backbone unfrozen)')
    ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()


## 8. Score the validation and test splits

Two splits, two different jobs, and the order matters.

**Validation** is where the confidence threshold gets chosen. It has already
influenced training through early stopping, so it cannot carry the reported
figure — but that is exactly what makes it the right place to fit one more
parameter.

**Test** has not influenced training, early stopping, or the threshold. It is
touched once, to report metrics at the threshold already frozen on validation.

Selecting the threshold on test and then reporting test metrics at that point
fits the operating point to the same images the result is claimed on, which
biases the reported precision upward. See `ml/NEXT-STEPS.md` section 1.

The CSVs written here are a sanity check. **The authoritative scores come from
`ml/score_split.py`**, which runs the images through
`backend/app/services/inference.py` — the code that actually serves S10.
Preprocessing differences between this notebook and the backend have shifted
individual scores by up to 0.46 before.

In [ ]:
loss, accuracy = classifier.evaluate(test_ds)
print(f'Test loss {loss:.4f}   accuracy {accuracy:.4f}')
print('Accuracy alone is not the result — see the threshold sweep.\n')

POSITIVE = CLASS_NAMES.index('smoking')


def score_split(dataset, split):
    """Write label,score for one split, in manifest order.

    The dataset is built with training=False, so it is not shuffled and rows
    line up with the manifest. Any change to build() that shuffles an
    evaluation split silently misaligns labels from scores.
    """
    probs = classifier.predict(dataset, verbose=0)
    positive = probs[:, POSITIVE]

    # select_threshold.py sweeps thresholds between 0 and 1, so anything
    # outside that range makes every threshold reject everything.
    assert positive.min() >= 0.0 and positive.max() <= 1.0, (
        f'Scores outside [0,1]: {positive.min():.3f} to {positive.max():.3f}. '
        'The model is emitting logits — re-run the model cell with softmax.')

    rows = manifest[manifest['split'] == split].reset_index(drop=True)
    assert len(rows) == len(positive), (
        f'{len(rows)} manifest rows but {len(positive)} scores for {split}.')

    frame = pd.DataFrame({'label': rows['label'], 'score': positive})
    path = f'/content/{split}_scores.csv'
    frame.to_csv(path, index=False)

    gap = (frame[frame.label == 'smoking']['score'].mean()
           - frame[frame.label == 'notsmoking']['score'].mean())
    print(f'{split}: {len(frame)} images, mean score gap {gap:.3f} '
          f'({"healthy" if gap > 0.3 else "WEAK — see notes below"})')
    return path


val_path = score_split(val_ds, 'val')
test_path = score_split(test_ds, 'test')

files.download(val_path)
files.download(test_path)
files.download('/content/training_curves.png')

print('\nThreshold: select on val_scores.csv, then report on test_scores.csv')
print('at that frozen value. Re-score through the backend first — see section 10.')

## 9. Export the model

Saved in Keras 3 format and loaded by `backend/app/services/inference.py`.
Download it, unzip into `ml/artifacts/`, and set `AI_MODEL_PATH` in
`backend/.env`.

Record `MODEL_VERSION` in `backend/.env` too — `AI_CLASSIFICATION.model_version`
stores it per report, so a later retrain never silently changes the meaning of
a historical confidence score.


In [ ]:
MODEL_VERSION = 'vit-scope-v1'

classifier.save(f'/content/{MODEL_VERSION}.keras')

metadata = {
    'model_version': MODEL_VERSION,
    'preset': PRESET,
    'image_size': IMAGE_SIZE,
    'class_names': CLASS_NAMES,
    'positive_index': CLASS_NAMES.index('smoking'),
    'test_accuracy': float(accuracy),
    'train_images': int((manifest['split'] == 'train').sum()),
    'val_images': int((manifest['split'] == 'val').sum()),
    'test_images': int((manifest['split'] == 'test').sum()),
}
pathlib.Path('/content/model_metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

files.download(f'/content/{MODEL_VERSION}.keras')
files.download('/content/model_metadata.json')


## 10. Next

1. Put `vit-scope-v1.keras` in `ml/artifacts/` and point `AI_MODEL_PATH` at it
2. Re-score both splits through the backend's own inference code — not the
   CSVs downloaded above, which came from this notebook's pipeline:

   ```bash
   ./backend/.venv/bin/python ml/score_split.py val
   ./backend/.venv/bin/python ml/score_split.py test
   ```

3. **Select** the threshold on validation, and freeze what it returns:

   ```bash
   ./backend/.venv/bin/python ml/select_threshold.py \
       ml/artifacts/val_scores_backend.csv
   ```

4. **Report** on test at that frozen threshold, changing nothing afterwards:

   ```bash
   ./backend/.venv/bin/python ml/select_threshold.py \
       ml/artifacts/test_scores_backend.csv --at-threshold <frozen>
   ```

5. Set `AI_CONFIDENCE_THRESHOLD` and `AI_MODEL_VERSION` in `backend/.env`

Steps 3 and 4 are not interchangeable. If the threshold is adjusted after
seeing step 4's numbers, the test split has been spent and the figure is no
longer an independent estimate.

Keep `training_curves.png`, the composition table from `ml/data/report.md`, and
both threshold outputs — together they are the evaluation evidence for
Chapter 5.